In [1]:
# Cell 1: Imports
import numpy as np
import os
import matplotlib.pyplot as plt
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Masking
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
import random

# Set seeds for reproducibility
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)
random.seed(SEED)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Num GPUs Available: {len(tf.config.experimental.list_physical_devices('GPU'))}")

TensorFlow Version: 2.16.2
Num GPUs Available: 0


In [2]:
# Cell 2: Configuration (No PCA)
PROCESSED_DATA_DIR = os.path.join(os.getcwd(), 'steinmetz_data_downloads')
os.makedirs(PROCESSED_DATA_DIR, exist_ok=True)
print(f"Data will be downloaded/loaded from: {PROCESSED_DATA_DIR}")

TARGET_BRAIN_REGION_X = 'VISp' # Input features from VISp
TARGET_BRAIN_REGION_Y = 'MOs'  # Target variable from MOs

# Time window for VISp LFP data (X), relative to an alignment event
X_TIME_WINDOW_START_POST_EVENT_MS = 0 
X_TIME_WINDOW_END_POST_EVENT_MS = 500 

# Time window for MOs LFP data (Y), relative to response_time
Y_TIME_WINDOW_BEFORE_RESPONSE_MS = 100 
Y_TIME_WINDOW_AFTER_RESPONSE_MS = 100 

# LSTM Hyperparameters
N_EPOCHS = 100 
BATCH_SIZE = 64
LSTM_UNITS_1 = 128
LSTM_UNITS_2 = 64
DENSE_UNITS_REG = 64 
DROPOUT_RATE = 0.3
LEARNING_RATE = 0.001 
N_SPLITS_CV = 5
EARLY_STOPPING_PATIENCE = 20

# PCA Configuration - All PCA is disabled
APPLY_PCA_X = False # Disabled
N_PCA_COMPONENTS_X = None # Not used
APPLY_PCA_Y = False # Disabled
N_PCA_COMPONENTS_Y = None # Not used

# Upsampling Configuration (Keep if needed, independent of PCA)
APPLY_UPSAMPLING = True # Or False, depending on your previous choice

Data will be downloaded/loaded from: /Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data_downloads


In [3]:
# Cell 3: Data Loading Functions
def load_steinmetz_data(data_dir):
    alldat = np.load(os.path.join(data_dir, 'alldat.npy'), allow_pickle=True)
    dat_LFP = np.load(os.path.join(data_dir, 'dat_LFP.npy'), allow_pickle=True)
    print(f"Loaded 'alldat' with {len(alldat)} sessions.")
    print(f"Loaded 'dat_LFP' with {len(dat_LFP)} sessions.")
    return alldat, dat_LFP

alldat, dat_LFP = load_steinmetz_data(PROCESSED_DATA_DIR)

FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/SSD/IMPACT SCHOLAR SUBMISSION 1/neural_navigators-main/notebooks/steinmetz_data_downloads/alldat.npy'

In [ ]:
# Cell 4: Helper Function - Mouse Details
mouse_age_data = {
    'Cori': 11, 'Forssmann': 26, 'Hench': 31, 'Lederberg': 15, 
    'Moniz': 30, 'Muller': 17, 'Radnitz': 20, 'Richards': 26, 
    'Tatum': 15, 'Theiler': 46 
}
YOUNGER_THRESHOLD_WEEKS = 20 

def get_mouse_details_by_session(session_idx, session_data_main):
    mouse_name = session_data_main.get('mouse_name', f"MouseUnknown_S{session_idx}")
    mouse_age_weeks = mouse_age_data.get(mouse_name)
    
    age_category = 'Unknown'
    if mouse_age_weeks is not None:
        if mouse_age_weeks <= YOUNGER_THRESHOLD_WEEKS:
            age_category = 'Younger'
        else:
            age_category = 'Older'
            
    return mouse_name, mouse_age_weeks, age_category

In [ ]:
# Cell 5: Data Preprocessing for MOs LFP Prediction (No PCA)
all_X_data = [] 
all_Y_data = [] 
all_session_ids = []
all_mouse_names = []
all_age_categories = []
all_trial_indices_original = [] 

print(f"Processing {len(alldat)} sessions...")

for session_idx in range(len(alldat)):
    session_data_main = alldat[session_idx]
    session_data_lfp = dat_LFP[session_idx]
    event_source_for_X = "stim_onset" 

    if session_idx == 0: # Debug prints for the first session
        print(f"
--- Debugging Session {session_idx} ---")
        # (Keep your existing debug prints here if you find them useful)
        print(f"--------------------------------------")

    mouse_name, mouse_age, age_category = get_mouse_details_by_session(session_idx, session_data_main)
    if age_category == 'Unknown':
        continue
        
    print(f'
--- Processing Session {session_idx} (Mouse: {mouse_name}, Age: {mouse_age}w, Category: {age_category}) ---')
    
    required_lfp_keys = ['lfp', 'brain_area_lfp']
    if any(key not in session_data_lfp for key in required_lfp_keys):
        print(f'  Warning: Session {session_idx} missing LFP keys. Skipping.')
        continue
        
    required_main_keys = ['stim_onset', 'response_time', 'bin_size', 'gocue']
    if any(key not in session_data_main for key in required_main_keys):
        print(f'  Warning: Session {session_idx} missing main data keys. Skipping.')
        continue

    lfp_all_channels = session_data_lfp['lfp'] 
    lfp_brain_areas = np.array(session_data_lfp['brain_area_lfp'])
    dt_lfp_s = session_data_main['bin_size'] 
    n_trials_lfp = lfp_all_channels.shape[1]

    response_times_orig = session_data_main['response_time']
    response_times = np.asarray(response_times_orig).squeeze()
    if response_times.ndim == 0: response_times = np.array([response_times.item()])
    n_trials_resp = len(response_times)

    event_times_for_X = None
    stim_onset_times_orig = session_data_main['stim_onset']
    stim_onset_times = np.asarray(stim_onset_times_orig).squeeze()
    if stim_onset_times.ndim == 0: stim_onset_times = np.array([stim_onset_times.item()])
    n_trials_stim = len(stim_onset_times)

    if n_trials_stim > 1 and (n_trials_stim == n_trials_lfp or n_trials_stim == n_trials_resp):
        event_times_for_X = stim_onset_times
        event_source_for_X = "stim_onset"
        print(f"  Using 'stim_onset' for X alignment ({n_trials_stim} events).")
    else:
        gocue_times_orig = session_data_main.get('gocue')
        if gocue_times_orig is not None:
            gocue_times = np.asarray(gocue_times_orig).squeeze()
            if gocue_times.ndim == 0: gocue_times = np.array([gocue_times.item()])
            n_trials_gocue = len(gocue_times)
            if n_trials_gocue > 1 and (n_trials_gocue == n_trials_lfp or n_trials_gocue == n_trials_resp):
                event_times_for_X = gocue_times
                event_source_for_X = "gocue"
                print(f"  'stim_onset' unsuitable, using 'gocue' for X alignment ({n_trials_gocue} events).")
            else:
                print(f"  'stim_onset' and 'gocue' unsuitable. Skipping session {session_idx}.")
                continue
        else:
            print(f"  'stim_onset' unsuitable and 'gocue' not found. Skipping session {session_idx}.")
            continue
            
    if event_times_for_X is None:
        print(f"  Error: No suitable event for X alignment. Skipping session {session_idx}.")
        continue

    n_trials_event_X = len(event_times_for_X)
    n_trials_session = min(n_trials_lfp, n_trials_event_X, n_trials_resp)

    if n_trials_session == 0:
        print(f'  Skipping session {session_idx}: Zero common trials.')
        continue
        
    if n_trials_lfp != n_trials_session: lfp_all_channels = lfp_all_channels[:, :n_trials_session, :]
    if n_trials_event_X != n_trials_session: event_times_for_X = event_times_for_X[:n_trials_session]
    if n_trials_resp != n_trials_session: response_times = response_times[:n_trials_session]
    
    print(f"  Using {n_trials_session} trials for session {session_idx} based on {event_source_for_X}, LFP, and response_time.")

    visp_channel_indices = np.where(lfp_brain_areas == TARGET_BRAIN_REGION_X)[0]
    if len(visp_channel_indices) == 0:
        print(f'  Warning: No LFP channels for {TARGET_BRAIN_REGION_X}. Skipping.')
        continue
    lfp_visp_all_trials = lfp_all_channels[visp_channel_indices, :, :] 
    
    x_start_s_rel_event = X_TIME_WINDOW_START_POST_EVENT_MS / 1000.0
    x_end_s_rel_event = X_TIME_WINDOW_END_POST_EVENT_MS / 1000.0
    
    mos_channel_indices = np.where(np.isin(lfp_brain_areas, [TARGET_BRAIN_REGION_Y, 'MOs', 'MOp']))[0]
    if len(mos_channel_indices) == 0:
        print(f'  Warning: No LFP channels for {TARGET_BRAIN_REGION_Y}. Skipping.')
        continue
    lfp_mos_all_trials = lfp_all_channels[mos_channel_indices, :, :] 
    
    y_window_before_s = Y_TIME_WINDOW_BEFORE_RESPONSE_MS / 1000.0
    y_window_after_s = Y_TIME_WINDOW_AFTER_RESPONSE_MS / 1000.0
    
    session_X_data_list = []
    session_Y_data_list = []
    session_trial_ids_list = []
    
    for trial_idx in range(n_trials_session): 
        event_t_x = event_times_for_X[trial_idx]
        response_t = response_times[trial_idx]
        
        if np.isnan(event_t_x) or np.isnan(response_t): continue
            
        x_window_start_abs_s = event_t_x + x_start_s_rel_event
        x_window_end_abs_s = event_t_x + x_end_s_rel_event
        x_start_lfp_idx = int(np.round(x_window_start_abs_s / dt_lfp_s))
        x_end_lfp_idx = int(np.round(x_window_end_abs_s / dt_lfp_s))
        
        if not (0 <= x_start_lfp_idx < x_end_lfp_idx <= lfp_visp_all_trials.shape[2]): continue
        x_trial_lfp = lfp_visp_all_trials[:, trial_idx, x_start_lfp_idx:x_end_lfp_idx]
        if x_trial_lfp.shape[1] == 0: continue
        
        y_window_start_abs_s = response_t - y_window_before_s
        y_window_end_abs_s = response_t + y_window_after_s
        y_start_lfp_idx = int(np.round(y_window_start_abs_s / dt_lfp_s))
        y_end_lfp_idx = int(np.round(y_window_end_abs_s / dt_lfp_s))
        
        if not (0 <= y_start_lfp_idx < y_end_lfp_idx <= lfp_mos_all_trials.shape[2]): continue
        y_trial_lfp = lfp_mos_all_trials[:, trial_idx, y_start_lfp_idx:y_end_lfp_idx]
        if y_trial_lfp.shape[1] == 0: continue

        session_X_data_list.append(x_trial_lfp.T) # Shape: (n_timesteps_x, n_visp_channels)
        session_Y_data_list.append(y_trial_lfp.T) # Shape: (n_timesteps_y, n_mos_channels)
        session_trial_ids_list.append(trial_idx)
        
    if not session_X_data_list: 
        print(f'  No valid trials processed for session {session_idx} after trial loop.')
        continue

    # --- X Data Processing (No PCA) ---
    max_len_x = max(arr.shape[0] for arr in session_X_data_list)
    X_session_padded = np.array([np.pad(arr, ((0, max_len_x - arr.shape[0]), (0,0)), 'constant', constant_values=0) for arr in session_X_data_list])
    X_session_processed = X_session_padded # Use raw padded data, no PCA

    # --- Y Data Processing (No PCA) ---
    max_len_y = max(arr.shape[0] for arr in session_Y_data_list)
    # Y_session_padded_raw has shape (n_trials_session, max_len_y, n_mos_channels_raw)
    Y_session_padded_raw = np.array([np.pad(arr, ((0, max_len_y - arr.shape[0]), (0,0)), 'constant', constant_values=0) for arr in session_Y_data_list])
    
    # Average across channels first, then average across time for Y
    if Y_session_padded_raw.shape[2] > 0: # Ensure there are channels to average
        Y_avg_channels = np.mean(Y_session_padded_raw, axis=2) # Shape: (n_trials_session, max_len_y)
    else: # Should not happen if mos_channel_indices is found
        Y_avg_channels = np.zeros((Y_session_padded_raw.shape[0], Y_session_padded_raw.shape[1]))

    Y_session_processed_target = np.mean(Y_avg_channels, axis=1).reshape(-1, 1) # Shape: (n_trials_session, 1)
            
    if X_session_processed.shape[0] == 0 or Y_session_processed_target.shape[0] == 0:
        print(f'  Session {session_idx} resulted in empty X or Y. Skipping.')
        continue
        
    all_X_data.append(X_session_processed)
    all_Y_data.append(Y_session_processed_target)
    num_valid_trials_session = X_session_processed.shape[0]
    all_session_ids.extend([session_idx] * num_valid_trials_session)
    all_mouse_names.extend([mouse_name] * num_valid_trials_session)
    all_age_categories.extend([age_category] * num_valid_trials_session)
    all_trial_indices_original.extend(session_trial_ids_list) 

    print(f'  Session {session_idx}: Added {num_valid_trials_session} trials. X shape: {X_session_processed.shape}, Y shape: {Y_session_processed_target.shape}')

if not all_X_data:
    raise ValueError("No data collected. Check preprocessing, brain regions, windowing, and event alignment.")

X_combined = np.concatenate(all_X_data, axis=0)
Y_combined = np.concatenate(all_Y_data, axis=0) 
groups = np.array(all_session_ids)
mouse_names_arr = np.array(all_mouse_names)
age_categories_arr = np.array(all_age_categories)

print(f'
Shape of X_combined before scaling: {X_combined.shape}') # X features = raw VISp channels
print(f'Shape of Y_combined: {Y_combined.shape}') # Y features = 1 (avg MOs LFP)

# Global Scaling for X features
if X_combined.size > 0 and X_combined.shape[2] > 0:
    n_trials_total_x, n_timesteps_x, n_features_x = X_combined.shape
    X_reshaped = X_combined.reshape(-1, n_features_x)
    global_x_scaler = StandardScaler()
    X_scaled_reshaped = global_x_scaler.fit_transform(X_reshaped)
    X_final = X_scaled_reshaped.reshape(n_trials_total_x, n_timesteps_x, n_features_x)
else:
    print("Warning: X_combined is empty or has no features. Scaling skipped.")
    X_final = X_combined 

print(f'Shape of final X after scaling: {X_final.shape}')

Y_final = Y_combined 
print(f'Shape of final Y: {Y_final.shape}')

unique_categories, counts = np.unique(age_categories_arr, return_counts=True)
print('
Trial counts per age category:')
for category, count in zip(unique_categories, counts):
    print(f'  {category}: {count}')

if X_final.shape[0] > 0:
    print(f'Total valid trials: {X_final.shape[0]}')
else:
    print('Total valid trials: 0')

In [ ]:
# Cell 6: Model Definition
def create_lstm_model(input_shape_x, output_dim_y, lstm_units_1, lstm_units_2, dense_units_reg, dropout_rate, learning_rate):
    model = Sequential([
        Masking(mask_value=0., input_shape=input_shape_x),
        LSTM(lstm_units_1, return_sequences=True),
        Dropout(dropout_rate),
        LSTM(lstm_units_2, return_sequences=False), # Last LSTM layer returns only the final output
        Dropout(dropout_rate),
        Dense(dense_units_reg, activation='relu'),
        Dropout(dropout_rate),
        Dense(output_dim_y, activation='linear') # Output layer for regression
    ])
    
    optimizer = Adam(learning_rate=learning_rate)
    model.compile(optimizer=optimizer, loss='mean_squared_error', metrics=['mae'])
    return model

# Example: Define input shape and output dim based on final processed data
# This will be used inside the CV loop after X_final and Y_final are defined for the whole dataset
# For now, just to show, if X_final and Y_final were available:
# if 'X_final' in globals() and X_final.size > 0 and 'Y_final' in globals() and Y_final.size > 0:
#     input_shape_X_glob = (X_final.shape[1], X_final.shape[2]) # (n_timesteps_x, n_visp_channels)
#     output_dim_Y_glob = Y_final.shape[1] # Should be 1
#     print(f"Example Model Input Shape: {input_shape_X_glob}, Output Dim: {output_dim_Y_glob}")
#     # model_example = create_lstm_model(input_shape_X_glob, output_dim_Y_glob, LSTM_UNITS_1, LSTM_UNITS_2, DENSE_UNITS_REG, DROPOUT_RATE, LEARNING_RATE)
#     # model_example.summary()
# else:
#     print("X_final/Y_final not ready for model shape example yet.")

In [ ]:
# Cell 7: Model Training, Cross-Validation, and Evaluation

if 'X_final' not in globals() or X_final.shape[0] == 0:
    print("X_final is not defined or is empty. Skipping model training and evaluation.")
else:
    gkf = GroupKFold(n_splits=N_SPLITS_CV)
    
    fold_val_mse = []
    fold_val_mae = []
    fold_val_r2 = []
    fold_histories = []
    
    all_y_true_val_folds = []
    all_y_pred_val_folds = []
    all_age_categories_val_folds = []

    # Determine input shape and output dim from the final dataset
    input_shape_X_dynamic = (X_final.shape[1], X_final.shape[2])
    output_dim_Y_dynamic = Y_final.shape[1] # Should be 1
    print(f"Using Input Shape: {input_shape_X_dynamic}, Output Dim: {output_dim_Y_dynamic} for training.")

    for fold, (train_idx, val_idx) in enumerate(gkf.split(X_final, Y_final, groups)):
        print(f'
--- Fold {fold+1}/{N_SPLITS_CV} ---')
        X_train, X_val = X_final[train_idx], X_final[val_idx]
        Y_train, Y_val = Y_final[train_idx], Y_final[val_idx]
        age_cat_val = age_categories_arr[val_idx]
        
        print(f'Shape of X_train: {X_train.shape}, Y_train: {Y_train.shape}')
        print(f'Shape of X_val: {X_val.shape}, Y_val: {Y_val.shape}')

        # Create a new model for each fold
        model = create_lstm_model(
            input_shape_X_dynamic, 
            output_dim_Y_dynamic,
            LSTM_UNITS_1, LSTM_UNITS_2, DENSE_UNITS_REG, 
            DROPOUT_RATE, LEARNING_RATE
        )
        if fold == 0: model.summary() # Print summary for the first fold

        early_stopping = EarlyStopping(
            monitor='val_loss', 
            patience=EARLY_STOPPING_PATIENCE, 
            verbose=1, 
            restore_best_weights=True
        )
        
        # Optional: Model checkpoint to save the best model of each fold
        # checkpoint_filepath = f'/tmp/fold_{fold+1}_best_model.keras'
        # model_checkpoint_callback = ModelCheckpoint(
        #     filepath=checkpoint_filepath,
        #     save_weights_only=False, # Set to True if you only want weights
        #     monitor='val_loss',
        #     mode='min',
        #     save_best_only=True)

        history = model.fit(
            X_train, Y_train,
            epochs=N_EPOCHS,
            batch_size=BATCH_SIZE,
            validation_data=(X_val, Y_val),
            callbacks=[early_stopping], # Add model_checkpoint_callback here if using
            verbose=1
        )
        fold_histories.append(history)
        
        Y_pred_val = model.predict(X_val)
        
        mse_val = mean_squared_error(Y_val, Y_pred_val)
        mae_val = mean_absolute_error(Y_val, Y_pred_val)
        r2_val = r2_score(Y_val, Y_pred_val)
        
        print(f'Fold {fold+1} Val MSE: {mse_val:.4f}, MAE: {mae_val:.4f}, R-squared: {r2_val:.4f}')
        
        fold_val_mse.append(mse_val)
        fold_val_mae.append(mae_val)
        fold_val_r2.append(r2_val)
        
        all_y_true_val_folds.append(Y_val)
        all_y_pred_val_folds.append(Y_pred_val)
        all_age_categories_val_folds.append(age_cat_val)

In [ ]:
# Cell 8: Results Aggregation and Age-Based Evaluation

if 'X_final' not in globals() or X_final.shape[0] == 0:
    print("X_final is not defined or is empty. Skipping results aggregation.")
elif not fold_val_mse: # Check if training actually ran
    print("No training folds completed. Skipping results aggregation.")
else:
    print("

--- Cross-Validation Summary ---")
    print(f"Average Validation MSE across {N_SPLITS_CV} folds: {np.nanmean(fold_val_mse):.4f} (+/- {np.nanstd(fold_val_mse):.4f}")
    print(f"Average Validation MAE across {N_SPLITS_CV} folds: {np.nanmean(fold_val_mae):.4f} (+/- {np.nanstd(fold_val_mae):.4f}")
    print(f"Average Validation R-squared across {N_SPLITS_CV} folds: {np.nanmean(fold_val_r2):.4f} (+/- {np.nanstd(fold_val_r2):.4f}")
    
    if fold_histories:
        plt.figure(figsize=(15, 5))
        for i, history_fold in enumerate(fold_histories):
            plt.plot(history_fold.history['val_loss'], label=f'Fold {i+1} Val Loss (MSE)')
        plt.title('Validation Loss (MSE) Across Folds')
        plt.xlabel('Epochs')
        plt.ylabel('MSE')
        plt.legend()
        plt.grid(True)
        plt.show()
    else:
        print("No training histories recorded to plot.")
        
    if all_y_true_val_folds and all_y_pred_val_folds and all_age_categories_val_folds:
        Y_true_all_cv = np.concatenate(all_y_true_val_folds, axis=0)
        Y_pred_all_cv = np.concatenate(all_y_pred_val_folds, axis=0)
        age_cat_all_cv = np.concatenate(all_age_categories_val_folds, axis=0)
        
        if Y_true_all_cv.shape[0] > 1:
            overall_r2 = r2_score(Y_true_all_cv, Y_pred_all_cv)
            print(f'
Overall R-squared on concatenated CV validation data: {overall_r2:.4f}')
        else:
            print(f'
Overall R-squared: Not enough samples (found {Y_true_all_cv.shape[0]})')

        print("
--- Performance by Age Category (on all CV validation samples) ---")
        for category in ['Younger', 'Older']:
            cat_indices = (age_cat_all_cv == category)
            if np.any(cat_indices):
                y_true_cat = Y_true_all_cv[cat_indices]
                y_pred_cat = Y_pred_all_cv[cat_indices]
                if len(y_true_cat) > 1:
                    mse_cat = mean_squared_error(y_true_cat, y_pred_cat)
                    mae_cat = mean_absolute_error(y_true_cat, y_pred_cat)
                    r2_cat = r2_score(y_true_cat, y_pred_cat)
                    print(f'  Category: {category} (Samples: {len(y_true_cat)})')
                    print(f'    MSE: {mse_cat:.4f}, MAE: {mae_cat:.4f}, R-squared: {r2_cat:.4f}')
                else:
                    print(f'  Category: {category} (Samples: {len(y_true_cat)}) - Not enough samples for R2.')
            else:
                print(f'  Category: {category} - No samples in validation set.')
                
        if Y_true_all_cv.shape[0] > 0:
            plt.figure(figsize=(12, 6))
            plot_samples = min(200, Y_true_all_cv.shape[0])
            # Since Y_true_all_cv is (n_samples, 1), we plot that single feature
            plt.plot(Y_true_all_cv[:plot_samples, 0], label='True Avg MOs LFP', alpha=0.7)
            plt.plot(Y_pred_all_cv[:plot_samples, 0], label='Predicted Avg MOs LFP', linestyle='--', alpha=0.7)
            plt.title('Overall True vs. Predicted Average MOs LFP (Validation Samples)')
            plt.xlabel('Sample Index')
            plt.ylabel('Avg LFP Value')
            plt.legend()
            plt.grid(True)
            plt.show()
        else:
            print("No validation samples to plot true vs. predicted.")
    else:
        print("No validation predictions recorded to aggregate or plot by age.")
        
print("
Script analysis section completed.")